# 04. Segunda prueba

Usamos las recuperaciones del experimento 03. En esta prueba cambiamos cómo se presentan los pasajes, cómo responde el modelo y cómo se revisan las citas.

Primero revisamos cobertura y salidas anteriores. Después generamos respuestas nuevas y comparamos las tres variantes sobre las mismas 50 preguntas.


## Preparación

En Colab, seleccionar GPU A100. Subir `data/colab/ai-week-v04.zip` a `Mi unidad/AIWEEK`, sin descomprimir. Abrir este notebook desde Drive.

En local, abrirlo desde el proyecto. Los resultados nuevos quedan en `experimentos/v04/`.


In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, platform, subprocess, sys, zipfile

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"
PAQUETE = "ai-week-v04.zip"

os.environ.update(USE_TF="0", USE_TORCH="1", USE_FLAX="0", TRANSFORMERS_NO_TF="1",
                  CUBLAS_WORKSPACE_CONFIG=":4096:8")
try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete = DRIVE / PAQUETE
    if not paquete.is_file():
        raise FileNotFoundError(f"Subir {PAQUETE} a {DRIVE}")
    RAIZ = Path("/content/aiweek") / ("v04-" + hash_archivo(paquete)[:12])
    if not (RAIZ / ".paquete_verificado").is_file():
        RAIZ.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (RAIZ / nombre).resolve().is_relative_to(RAIZ.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(RAIZ)
        inventario = json.loads((RAIZ / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(RAIZ / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (RAIZ / ".paquete_verificado").write_text("ok")
    if not (RAIZ / "scripts/v04/generacion.py").is_file():
        raise FileNotFoundError("El ZIP no contiene scripts/v04. Regenerarlo")
    dependencias = RAIZ / "requirements-colab.txt"
    firma = hash_archivo(dependencias) + platform.python_version()
    marca = Path("/content/aiweek_experimentos_dependencias.txt")
    if not marca.exists() or marca.read_text() != firma:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        marca.write_text(firma)
    RESULTADOS = DRIVE / "experimentos"
    CACHE_MODELOS = DRIVE / "modelos_cache"
    os.environ["HF_HOME"] = str(RAIZ / "data/cache_huggingface")
else:
    RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/v04").is_dir()), None)
    if RAIZ is None:
        raise FileNotFoundError("Abrir el notebook desde el proyecto")
    RESULTADOS = RAIZ / "data/experimentos_sistema"
    CACHE_MODELOS = None

os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Entorno:", "Colab" if EN_COLAB else "local", "| Resultados:", RESULTADOS)

## Ejecución anterior

Se usa la última ejecución del 03 y su recuperación. Si hace falta otra, escribir los nombres de las carpetas en la siguiente celda.


In [ ]:
import pandas as pd
from IPython.display import display
from scripts.v04 import experimento as v04

ejecucion_03 = ""
recuperacion_03 = ""
variantes = ["densa", "hibrida", "densa_reranker"]

puntero = json.loads((RESULTADOS / "ultima_ejecucion.json").read_text(encoding="utf-8"))
CARPETA_03 = RESULTADOS / "ejecuciones" / (ejecucion_03 or Path(puntero["carpeta"]).name)
config_03 = json.loads((CARPETA_03 / "configuracion.json").read_text(encoding="utf-8"))
CARPETA_RECUP = RESULTADOS / "recuperaciones" / (recuperacion_03 or Path(config_03["recuperacion"]).name)
print("03:", CARPETA_03.name, "| recuperación:", CARPETA_RECUP.name)
resumen_03 = pd.read_csv(CARPETA_03 / "resumen.csv").set_index("variante")
display(resumen_03[["total_50", "cerradas_20", "citas_20", "abstencion_10", "abstenciones", "errores_esquema"]])

## Cobertura del corpus

Revisamos si las normas de referencia están en el corpus y si llegaron a los pasajes del 03. Una norma ausente no se arregla cambiando el prompt.

La tabla del banco muestra prioridades para ampliar el corpus. No cambia los archivos ni el índice.


In [ ]:
from scripts.v04.cobertura import auditar, auditar_banco, indice_corpus

preguntas, entradas = v04.muestra(RAIZ)
indice = indice_corpus(RAIZ)
recups = {}
ev = v04.evidencia(RAIZ)
for variante in variantes:
    recups[variante] = {i: ev.reconstruir(p)[0] for i, p in v04.recuperacion_03(CARPETA_RECUP, variante).items()}
cobertura = pd.DataFrame(auditar(RAIZ, preguntas, recuperaciones=recups, indice=indice))
con_cita = cobertura[cobertura["estado_corpus"] != "sin_cita_extraible"]
print("Citas de referencia en la muestra:", len(con_cita))
display(pd.concat({v: con_cita[v].value_counts() for v in variantes}, axis=1).fillna(0).astype(int))
display(con_cita.groupby("area")["estado_corpus"].value_counts().unstack(fill_value=0))

banco = pd.DataFrame(auditar_banco(RAIZ, indice=indice))
total = banco["items_del_banco"].sum()
print(f"Banco de 992: {banco.loc[banco.en_corpus, 'items_del_banco'].sum()}/{total} menciones de norma cubiertas")
display(banco[~banco.en_corpus].sort_values("items_del_banco", ascending=False).head(15)[["norma", "items_del_banco", "areas", "donde_buscar"]])

## Revisar las salidas del 03

Aplicamos las reglas nuevas al texto ya generado, sin usar GPU. Esto permite medir qué aporta la revisión de formato y citas antes de volver a generar.


In [ ]:
SALIDA_REPUNTUADA = RESULTADOS / "v04" / ("repuntuado_" + CARPETA_03.name)
repuntuado = v04.repuntuar_03(RAIZ, CARPETA_03, CARPETA_RECUP, variantes, SALIDA_REPUNTUADA)
tabla_rep = pd.DataFrame(repuntuado).T
tabla_rep.insert(0, "total_03", resumen_03.loc[tabla_rep.index, "total_50"])
display(tabla_rep[["total_03", "total_50", "cerradas_20", "citas_20", "abstencion_10", "abstenciones", "tasa_sin_respaldo", "errores_esquema"]])
for variante in variantes:
    notas = pd.read_json(SALIDA_REPUNTUADA / variante / "notas.json")
    print(variante, notas["motivo"].value_counts().to_dict())

## Generar otra vez

Primero corremos seis preguntas con fallos conocidos. Si sus respuestas salen bien, ejecutamos las 50 preguntas en cada variante. Cada respuesta se guarda al terminar.

La celda muestra la GPU real. En Colab se detiene si no es una A100.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Seleccionar GPU A100 en Colab o una GPU NVIDIA en local")

gpu_actual = torch.cuda.get_device_name(0)
memoria_gpu = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu_actual} | memoria: {memoria_gpu:.0f} GiB")
if EN_COLAB and "A100" not in gpu_actual.upper():
    raise RuntimeError("Colab no asignó una A100. Cambiar el tipo de GPU antes de generar")

etiqueta_gpu = "a100" if "A100" in gpu_actual.upper() else "local"
config = {
    "decoder": "salamandra-7b-instruct",
    "max_tokens": 900,
    "repeat_penalty": 1.1,
    "politica": {"citar_evidencia": "todas", "abstener_libre": "sin_evidencia"},
}
prueba_ids = [51, 308, 24, 168, 472, 919]
carpeta_prueba = v04.ejecutar(RAIZ, CARPETA_RECUP, RESULTADOS, ["densa"], config=config,
                              cache_modelos=CACHE_MODELOS, nombre=f"prueba_salamandra_{etiqueta_gpu}",
                              solo_ids=prueba_ids)
for i in prueba_ids:
    fila = json.loads((carpeta_prueba / "densa" / "respuestas" / f"{i}.json").read_text(encoding="utf-8"))
    respuesta, registro = fila["respuesta"], fila["registro"]
    print(f"\n{i} | {respuesta['formato']} | fin: {registro.get('finish_reason')} | "
          f"tokens: {registro.get('tokens_salida')} | reparado: {registro.get('reparado')}")
    display({k: v for k, v in respuesta.items() if k != "pasajes_recuperados"})
    if fila["postproceso"]["oraciones_eliminadas"]:
        print("Oraciones eliminadas por citas sin respaldo:", fila["postproceso"]["oraciones_eliminadas"])


In [ ]:
nombre_04 = f"salamandra_v04_{etiqueta_gpu}"
CARPETA_04 = v04.ejecutar(RAIZ, CARPETA_RECUP, RESULTADOS, variantes, config=config,
                          cache_modelos=CACHE_MODELOS, nombre=nombre_04)
print("Guardado en", CARPETA_04)

## Comparar con el 03

La tabla muestra puntos, citas, abstenciones y preguntas que mejoraron o empeoraron. Con 50 preguntas, revisar los casos concretos antes de elegir una variante.


In [ ]:
resumen_04 = pd.DataFrame({v: json.loads((CARPETA_04 / v / "resumen.json").read_text()) for v in variantes}).T
resumen_04.insert(0, "total_03", resumen_03.loc[resumen_04.index, "total_50"])
display(resumen_04[["total_03", "total_50", "cerradas_20", "aciertos_cerradas", "citas_20", "recall_citas",
                    "tasa_sin_respaldo", "abstencion_10", "abstenciones", "errores_esquema"]])

pareadas = {v: v04.comparar(RAIZ, CARPETA_03, CARPETA_04, v) for v in variantes}
display(pd.concat({v: t["cambio"].value_counts() for v, t in pareadas.items()}, axis=1).fillna(0).astype(int))

## Fallos por área

La gráfica muestra tres señales distintas: cerradas incorrectas, citas de referencia no encontradas en la respuesta y abstenciones. Cada barra indica fallos sobre preguntas evaluables de esa área.

La muestra tiene pocas preguntas por área. El juez oficial no da corrección de texto libre por pregunta, así que esta gráfica no mide toda la calidad jurídica.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variante_area = resumen_04["total_50"].idxmax()
ruta_diagnostico = CARPETA_04 / variante_area / "diagnostico.json"
detalle_area = pd.DataFrame(json.loads(ruta_diagnostico.read_text(encoding="utf-8")))
detalle_area["cerrada"] = detalle_area["formato"].eq("multiple_choice")
detalle_area["cerrada_fallida"] = detalle_area["cerrada"] & ~detalle_area["acierto_cerrada"].fillna(False).astype(bool)
detalle_area["cita_evaluable"] = detalle_area["ref_citas"].fillna(0).gt(0)
detalle_area["cita_fallida"] = detalle_area["cita_evaluable"] & detalle_area["citas_aciertos"].fillna(0).eq(0)

areas = detalle_area.groupby("area").agg(
    preguntas=("id", "size"),
    cerradas=("cerrada", "sum"),
    cerradas_fallidas=("cerrada_fallida", "sum"),
    con_cita=("cita_evaluable", "sum"),
    sin_cita_correcta=("cita_fallida", "sum"),
    abstenciones=("abstencion", "sum"),
)
areas["tasa_citas"] = np.divide(
    areas["sin_cita_correcta"], areas["con_cita"].replace(0, np.nan)
)
areas = areas.sort_values(["tasa_citas", "abstenciones"], ascending=False)
print("Variante:", variante_area)
display(areas.drop(columns="tasa_citas"))

nombres = [a.replace("Derecho de los mercados [competencia, consumidor, datos personales y propiedad intelectual]", "Mercados")
             .replace("Derecho de ", "").replace("Derecho ", "").capitalize() for a in areas.index]
posiciones = np.arange(len(areas))
fig, ejes = plt.subplots(1, 3, figsize=(17, max(5, 0.55 * len(areas))), sharey=True)
metricas = [
    ("cerradas_fallidas", "cerradas", "Cerradas incorrectas", "#a76363"),
    ("sin_cita_correcta", "con_cita", "Sin cita de referencia", "#b68649"),
    ("abstenciones", "preguntas", "Abstenciones", "#6484a1"),
]
for eje, (col_fallos, col_total, titulo, color) in zip(ejes, metricas):
    fallos = areas[col_fallos].to_numpy(dtype=float)
    totales = areas[col_total].to_numpy(dtype=float)
    tasas = np.divide(fallos, totales, out=np.zeros_like(fallos), where=totales > 0) * 100
    eje.barh(posiciones[totales > 0], tasas[totales > 0], color=color)
    for pos, tasa, fallo, total in zip(posiciones, tasas, fallos, totales):
        etiqueta = f"{int(fallo)}/{int(total)}" if total else "sin datos"
        eje.text(min(tasa + 2, 102) if total else 2, pos, etiqueta, va="center", fontsize=9)
    eje.set_title(titulo)
    eje.set_xlim(0, 120)
    eje.set_xlabel("% de preguntas evaluables")
    eje.set_yticks(posiciones, nombres)
ejes[0].invert_yaxis()
fig.suptitle(f"Fallos por área · {variante_area}")
fig.subplots_adjust(left=0.21, right=0.98, top=0.87, bottom=0.12, wspace=0.12)
plt.show()


In [ ]:
variante_revisar = "densa"
t = pareadas[variante_revisar]
display(t[t["cambio"].isin(["rota", "igual_mal"])])

## Otro decoder

La comparación con Qwen2.5-7B está disponible. Se deja desactivada hasta revisar los resultados de Salamandra.


In [ ]:
correr_otro_decoder = False
if correr_otro_decoder:
    config_qwen = {**config, "decoder": "qwen2.5-7b-instruct"}
    CARPETA_QWEN = v04.ejecutar(RAIZ, CARPETA_RECUP, RESULTADOS, variantes, config=config_qwen,
                                cache_modelos=CACHE_MODELOS, nombre="qwen25_v04")
    display(pd.DataFrame({v: json.loads((CARPETA_QWEN / v / "resumen.json").read_text()) for v in variantes}).T)

## Juez oficial

La llave solo se usa para ejecutar el evaluador oficial. En Colab, guardarla en Secretos con el nombre `OPENROUTER_API_KEY` y dar acceso al notebook. En local, usar una variable de entorno o `scripts/.env`.

Se elige la variante con más puntos sin juez. La evaluación se guarda para no repetir llamadas pagadas. Si cambian las respuestas, activar `repetir_juez`. El juez entrega un puntaje total de texto libre, no resultados por área.


In [ ]:
usar_juez = True
repetir_juez = False
variante_juez = resumen_04["total_50"].idxmax()
carpeta_finalista = CARPETA_04 / variante_juez
salida_juez = carpeta_finalista / "evaluacion_con_ragas.json"

if usar_juez:
    if repetir_juez or not salida_juez.is_file():
        if EN_COLAB:
            from google.colab import userdata
            llave = userdata.get("OPENROUTER_API_KEY")
        else:
            llave = os.environ.get("OPENROUTER_API_KEY")
            if not llave:
                for ruta_env in (RAIZ / "scripts/.env", RAIZ / "data/oficial/scripts/.env"):
                    if ruta_env.is_file():
                        for linea in ruta_env.read_text(encoding="utf-8").splitlines():
                            if linea.strip().startswith("OPENROUTER_API_KEY="):
                                llave = linea.split("=", 1)[1].strip().strip('"').strip("'")
                                break
                    if llave:
                        break
        if not llave:
            raise RuntimeError("Falta OPENROUTER_API_KEY en Secretos de Colab o en el entorno local")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(RAIZ / "data/oficial/scripts/requirements-evaluador.txt")], check=True)
        entorno_juez = {**os.environ, "OPENROUTER_API_KEY": llave}
        comando_juez = [sys.executable, str(RAIZ / "data/oficial/scripts/evaluate.py"),
                        "--submission", str(carpeta_finalista / "submissions.jsonl"),
                        "--split", "sample", "--ragas", "--out", str(salida_juez)]
        proceso_juez = subprocess.run(comando_juez, cwd=RAIZ / "data/oficial",
                                     env=entorno_juez, capture_output=True, text=True)
        del llave, entorno_juez
        if proceso_juez.returncode:
            raise RuntimeError("Falló el evaluador oficial: " + proceso_juez.stderr[-2000:])
    reporte_juez = json.loads(salida_juez.read_text(encoding="utf-8"))
    print("Variante evaluada:", variante_juez)
    display(pd.DataFrame([{
        "cerradas_20": reporte_juez["cerradas"]["puntos"],
        "citas_20": reporte_juez["citas"]["puntos"],
        "abstencion_10": reporte_juez["abstencion"]["puntos"],
        "texto_libre_30": reporte_juez["correccion_ragas"]["puntos"],
        "total_80": reporte_juez["total_automatico"]["obtenidos"],
    }]))


## Resultados y decisiones

Pendiente de esta ejecución. Revisar el puntaje completo, las preguntas que cambiaron y la gráfica por área antes de elegir la variante final.
